
### SER-01. Final Model Artifact Serialization

**Objective:** Package the selected preprocessing pipeline, trained XGBoost classifier, decision threshold, and feature engineering logic into a reusable inference bundle.

**Input:**
- Fitted Tree Preprocessor from the preprocessing stage.
- Selected XGBoost with Class Weight model from RQ2.
- Selected decision threshold from RQ3.
- Shared feature engineering module.

**Procedure:** Verify the frozen model configuration, copy the existing trained artifacts into the Final model directory, and record the feature schema, software versions, and artifact checksums.

**Output:** A versioned inference bundle containing the final preprocessor, trained classifier, threshold configuration, feature engineering module, and model manifest.

**Methodological consideration:** No model fitting, preprocessing fitting, threshold selection, or test-based optimization is performed. The artifacts are preserved from their respective completed stages.


In [1]:

# ============================================================
# SER-01 — FINAL MODEL ARTIFACT SERIALIZATION
# ============================================================

from google.colab import drive
from pathlib import Path

import hashlib
import json
import shutil
import sys
import platform

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost

# ------------------------------------------------------------
# STEP 1. KẾT NỐI GOOGLE DRIVE
# ------------------------------------------------------------

drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/Shareddrives/"
    "M(anchine) L(earning)/Output"
)

EXP = ROOT / "ML_Experiments"

PP_DIR = ROOT / "Preprocessing"
RQ2_DIR = EXP / "RQ2"
RQ3_DIR = EXP / "RQ3"
FINAL_DIR = EXP / "Final"

MODEL_DIR = FINAL_DIR / "model"
RESULT_DIR = FINAL_DIR / "results"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# STEP 2. ĐỌC CẤU HÌNH RQ2 VÀ RQ3
# ------------------------------------------------------------

rq2_config_path = (
    RQ2_DIR / "results" / "selected_strategy.json"
)

rq3_config_path = (
    RQ3_DIR / "results" / "selected_threshold.json"
)

assert rq2_config_path.is_file()
assert rq3_config_path.is_file()

with open(rq2_config_path, encoding="utf-8") as f:
    rq2_config = json.load(f)

with open(rq3_config_path, encoding="utf-8") as f:
    rq3_config = json.load(f)

assert rq2_config["selected_algorithm"] == "XGBoost"
assert rq2_config["selected_strategy"] == "Class_Weight"

assert rq3_config["selected_algorithm"] == "XGBoost"
assert (
    rq3_config["selected_imbalance_strategy"]
    == "Class_Weight"
)

assert rq3_config["selection_metric"] == "F1"
assert rq3_config["test_used_for_selection"] is False

FINAL_THRESHOLD = float(
    rq3_config["selected_threshold"]
)

assert np.isfinite(FINAL_THRESHOLD)
assert 0 <= FINAL_THRESHOLD <= 1

# ------------------------------------------------------------
# STEP 3. XÁC ĐỊNH ARTIFACT GỐC
# ------------------------------------------------------------

source_model = Path(
    rq2_config["selected_model_path"]
)

source_preprocessor = Path(
    rq2_config["preprocessor_path"]
)

# Module Feature Engineering đã được lưu ở PP06.
source_feature_module = (
    PP_DIR / "fraud_preprocessing.py"
)

assert source_model.is_file()
assert source_preprocessor.is_file()
assert source_feature_module.is_file()

# Xác nhận RQ3 tham chiếu cùng mô hình RQ2.
assert Path(rq3_config["model_path"]) == source_model

assert (
    Path(rq3_config["preprocessor_path"])
    == source_preprocessor
)

# ------------------------------------------------------------
# STEP 4. COPY ARTIFACTS SANG FINAL/MODEL
# ------------------------------------------------------------

final_model_path = (
    MODEL_DIR / "final_model.joblib"
)

final_preprocessor_path = (
    MODEL_DIR / "final_preprocessor.joblib"
)

final_feature_module_path = (
    MODEL_DIR / "fraud_preprocessing.py"
)

final_threshold_path = (
    MODEL_DIR / "selected_threshold.json"
)

shutil.copy2(
    source_model,
    final_model_path
)

shutil.copy2(
    source_preprocessor,
    final_preprocessor_path
)

shutil.copy2(
    source_feature_module,
    final_feature_module_path
)

shutil.copy2(
    rq3_config_path,
    final_threshold_path
)

# ------------------------------------------------------------
# STEP 5. KIỂM TRA CHECKSUM
# ------------------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


# Copy phải giữ nguyên nội dung artifact.
assert (
    sha256_file(source_model)
    == sha256_file(final_model_path)
)

assert (
    sha256_file(source_preprocessor)
    == sha256_file(final_preprocessor_path)
)

assert (
    sha256_file(source_feature_module)
    == sha256_file(final_feature_module_path)
)

# ------------------------------------------------------------
# STEP 6. LOAD KIỂM TRA SCHEMA
# ------------------------------------------------------------

saved_preprocessor = joblib.load(
    final_preprocessor_path
)

saved_model = joblib.load(
    final_model_path
)

assert hasattr(saved_preprocessor, "transform")
assert hasattr(saved_model, "predict_proba")

assert set(saved_model.classes_) == {0, 1}

output_features = list(
    saved_preprocessor.get_feature_names_out()
)

assert len(output_features) == 13

assert saved_model.n_features_in_ == 13

if hasattr(saved_model, "feature_names_in_"):
    assert (
        list(saved_model.feature_names_in_)
        == output_features
    )

# ------------------------------------------------------------
# STEP 7. LƯU MODEL MANIFEST
# ------------------------------------------------------------

manifest = {
    "artifact_name": "Final_Fraud_Detection_Inference_Bundle",
    "model_algorithm": "XGBoost",
    "imbalance_strategy": "Class_Weight",
    "preprocessor_family": "Tree",

    "selection_metric": "Maximum_Validation_F1",
    "selected_threshold": FINAL_THRESHOLD,

    "input_feature_count": 7,
    "engineered_feature_count": 9,
    "transformed_feature_count": 13,

    "output_feature_names": output_features,

    "files": {
        "model": final_model_path.name,
        "preprocessor": final_preprocessor_path.name,
        "feature_engineering": final_feature_module_path.name,
        "threshold": final_threshold_path.name
    },

    "sha256": {
        "model": sha256_file(final_model_path),
        "preprocessor": sha256_file(final_preprocessor_path),
        "feature_engineering": sha256_file(
            final_feature_module_path
        ),
        "threshold": sha256_file(final_threshold_path)
    },

    "software_versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgboost.__version__,
        "joblib": joblib.__version__
    },

    "training_status": "Previously_Trained",
    "model_retrained": False,
    "threshold_reoptimized": False,

    "intended_use": (
        "Post-transaction fraud scoring "
        "and review alert generation"
    )
}

manifest_path = (
    MODEL_DIR / "model_manifest.json"
)

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# STEP 8. FINAL CHECK
# ------------------------------------------------------------

required_outputs = [
    final_model_path,
    final_preprocessor_path,
    final_feature_module_path,
    final_threshold_path,
    manifest_path
]

for path in required_outputs:
    assert path.is_file()
    assert path.stat().st_size > 0

print("=" * 60)
print("SER-01 — FINAL MODEL SERIALIZATION")
print("=" * 60)

print("Algorithm: XGBoost")
print("Strategy: Class_Weight")
print("Threshold:", FINAL_THRESHOLD)

print("\nSerialized artifacts:")

for path in required_outputs:
    print("-", path.name)

print("\nTransformed features:", len(output_features))
print("Model feature count:", saved_model.n_features_in_)

print("\nPASS: SER-01 completed.")


Mounted at /content/drive
SER-01 — FINAL MODEL SERIALIZATION
Algorithm: XGBoost
Strategy: Class_Weight
Threshold: 0.9941363334655762

Serialized artifacts:
- final_model.joblib
- final_preprocessor.joblib
- fraud_preprocessing.py
- selected_threshold.json
- model_manifest.json

Transformed features: 13
Model feature count: 13

PASS: SER-01 completed.



### SER-02. Serialized Model Reload and Prediction Consistency

**Objective:** Verify that the serialized inference artifacts preserve the prediction behavior of the previously trained model and fitted preprocessor.

**Input:** Original RQ2 model, original Tree Preprocessor, serialized Final model and preprocessor, frozen decision threshold, and a deterministic sample from the Validation dataset.

**Procedure:** Reload the original and packaged artifacts, transform identical Validation records, generate fraud scores, and compare the resulting scores and binary alert decisions.

**Output:** A prediction consistency verification result and summary statistics.

**Verification criteria:**
- Identical transformed feature names and ordering.
- Numerically equivalent transformed features.
- Equivalent fraud scores within a strict floating-point tolerance.
- Exactly identical binary alert decisions.

**Methodological consideration:** Validation is used solely for serialization verification. No model fitting, threshold optimization, or performance-based configuration changes are performed.


In [2]:

# ============================================================
# SER-02 — RELOAD AND PREDICTION CONSISTENCY
# ============================================================

# ------------------------------------------------------------
# STEP 1. CHỌN MẪU VALIDATION
# ------------------------------------------------------------

validation_path = (
    PP_DIR / "3.2.7_Tree_Validation_20pct.parquet"
)

assert validation_path.is_file()

# Lấy số dòng nhỏ để kiểm tra kỹ thuật.
validation_df = pd.read_parquet(
    validation_path
)

assert "isFraud" in validation_df.columns
assert validation_df.index.is_unique

# Lấy cả fraud và non-fraud.
fraud_sample = validation_df.loc[
    validation_df["isFraud"] == 1
].sample(
    n=128,
    random_state=42
)

normal_sample = validation_df.loc[
    validation_df["isFraud"] == 0
].sample(
    n=128,
    random_state=42
)

verification_df = pd.concat(
    [fraud_sample, normal_sample]
).sort_index()

X_verify = verification_df.drop(
    columns=["isFraud"]
)

assert X_verify.shape == (256, 13)

# ------------------------------------------------------------
# STEP 2. LOAD BẢN GỐC VÀ BẢN SERIALIZED
# ------------------------------------------------------------

original_model = joblib.load(
    source_model
)

original_preprocessor = joblib.load(
    source_preprocessor
)

reloaded_model = joblib.load(
    final_model_path
)

reloaded_preprocessor = joblib.load(
    final_preprocessor_path
)

# ------------------------------------------------------------
# STEP 3. XÁC MINH CẤU TRÚC FEATURES
# ------------------------------------------------------------

original_columns = list(
    original_preprocessor.get_feature_names_out()
)

reloaded_columns = list(
    reloaded_preprocessor.get_feature_names_out()
)

assert original_columns == reloaded_columns
assert list(X_verify.columns) == original_columns

assert (
    original_model.n_features_in_
    == reloaded_model.n_features_in_
    == len(original_columns)
)

# ------------------------------------------------------------
# STEP 4. SO SÁNH FRAUD SCORES
# ------------------------------------------------------------

positive_class_original = list(
    original_model.classes_
).index(1)

positive_class_reloaded = list(
    reloaded_model.classes_
).index(1)

original_scores = original_model.predict_proba(
    X_verify
)[:, positive_class_original]

reloaded_scores = reloaded_model.predict_proba(
    X_verify
)[:, positive_class_reloaded]

assert original_scores.shape == reloaded_scores.shape

# Dung sai cho tính toán số thực.
SCORE_TOLERANCE = 1e-8

score_difference = np.abs(
    original_scores - reloaded_scores
)

maximum_difference = float(
    np.max(score_difference)
)

assert np.allclose(
    original_scores,
    reloaded_scores,
    rtol=0,
    atol=SCORE_TOLERANCE
), "Serialized model produces different fraud scores."

# ------------------------------------------------------------
# STEP 5. SO SÁNH QUYẾT ĐỊNH CẢNH BÁO
# ------------------------------------------------------------

original_alerts = (
    original_scores >= FINAL_THRESHOLD
).astype(np.int8)

reloaded_alerts = (
    reloaded_scores >= FINAL_THRESHOLD
).astype(np.int8)

assert np.array_equal(
    original_alerts,
    reloaded_alerts
), "Alert decisions changed after serialization."

# ------------------------------------------------------------
# STEP 6. HIỂN THỊ KẾT QUẢ
# ------------------------------------------------------------

consistency_df = pd.DataFrame(
    {
        "Original_Score": original_scores,
        "Reloaded_Score": reloaded_scores,
        "Absolute_Difference": score_difference,
        "Original_Alert": original_alerts,
        "Reloaded_Alert": reloaded_alerts
    },
    index=X_verify.index
)

display(
    consistency_df.head(10)
)

print("=" * 60)
print("SER-02 — MODEL RELOAD CONSISTENCY")
print("=" * 60)

print("Verification records:", len(X_verify))
print("Maximum score difference:", maximum_difference)

print(
    "Identical alert decisions:",
    bool(np.array_equal(
        original_alerts,
        reloaded_alerts
    ))
)

print("\nPASS: SER-02 completed.")


,Original_Score,Reloaded_Score,Absolute_Difference,Original_Alert,Reloaded_Alert
6702,0.999998,0.999998,0.0,1,1
33473,0.999997,0.999997,0.0,1,1
131979,0.000178,0.000178,0.0,0,0
169998,0.031124,0.031124,0.0,0,0
223234,0.000006,0.000006,0.0,0,0
259844,0.999994,0.999994,0.0,1,1
259883,0.000002,0.000002,0.0,0,0
301802,0.999648,0.999648,0.0,1,1
314754,0.000019,0.000019,0.0,0,0
403793,0.000003,0.000003,0.0,0,0


SER-02 — MODEL RELOAD CONSISTENCY
Verification records: 256
Maximum score difference: 0.0
Identical alert decisions: True

PASS: SER-02 completed.



### SER-03. End-to-End Inference Pipeline Verification

**Objective:** Verify that the serialized deployment bundle reproduces the established machine learning inference workflow from original transaction attributes to fraud alert decisions.

**Input:** Frozen serialized artifacts, shared feature engineering logic, and corresponding raw and preprocessed Validation records.

**Procedure:** Select deterministic Validation transactions, reconstruct engineered features using the packaged feature engineering module, apply the saved Tree Preprocessor, generate XGBoost fraud scores, and compare them against predictions generated from the previously preprocessed Validation data.

**Output:** End-to-end inference verification results and `serialization_verification.json`.

**Verification criteria:**
- Raw input consists of the seven expected transaction attributes.
- Feature engineering produces nine features.
- Tree preprocessing produces 13 features in the expected order.
- Preprocessed features match the established Validation preprocessing output.
- Fraud scores are numerically consistent.
- Binary alert decisions are identical.
- Invalid input does not bypass validation.

**Methodological consideration:** This is a functional and reproducibility check, not an additional model evaluation or a new threshold selection experiment.


In [3]:

# ============================================================
# SER-03 — END-TO-END INFERENCE VERIFICATION
# ============================================================

import importlib.util

# ------------------------------------------------------------
# STEP 1. LOAD MODULE FEATURE ENGINEERING TỪ BUNDLE
# ------------------------------------------------------------

module_spec = importlib.util.spec_from_file_location(
    "fraud_preprocessing",
    final_feature_module_path
)

fp = importlib.util.module_from_spec(module_spec)

module_spec.loader.exec_module(fp)

assert len(fp.BASE_FEATURES) == 7
assert len(fp.FEATURES) == 9

# ------------------------------------------------------------
# STEP 2. LOAD RAW VALIDATION DATA
# ------------------------------------------------------------

raw_validation_path = (
    ROOT
    / "Train_Validation_Test"
    / "3.2.8_Validation_20pct.parquet"
)

assert raw_validation_path.is_file()

# Chỉ đọc 7 trường dữ liệu gốc.
raw_validation = pd.read_parquet(
    raw_validation_path,
    columns=fp.BASE_FEATURES
)

assert raw_validation.index.is_unique

# Dùng chính các transaction indices của SER-02.
assert X_verify.index.difference(
    raw_validation.index
).empty

raw_sample = raw_validation.loc[
    X_verify.index,
    fp.BASE_FEATURES
]

# ------------------------------------------------------------
# STEP 3. TÁI HIỆN ĐẦY ĐỦ PIPELINE FLASK
# ------------------------------------------------------------

# Chuyển dữ liệu thành danh sách giao dịch.
records = raw_sample.to_dict(
    orient="records"
)

# Feature Engineering và preprocessing.
transformed_features = fp.transform_for_inference(
    records,
    reloaded_preprocessor,
    expected_columns=list(X_verify.columns)
)

# Khôi phục transaction index để đối chiếu.
transformed_features.index = X_verify.index

assert transformed_features.shape == (256, 13)

# ------------------------------------------------------------
# STEP 4. ĐỐI CHIẾU FEATURES
# ------------------------------------------------------------

# Kiểm tra kết quả của toàn bộ luồng 7 -> 9 -> 13.
# Phải khớp dữ liệu mà notebook đã dùng để đánh giá.

np.testing.assert_allclose(
    transformed_features.to_numpy(dtype=float),
    X_verify.to_numpy(dtype=float),
    rtol=1e-10,
    atol=1e-6
)

assert (
    list(transformed_features.columns)
    == list(X_verify.columns)
)

# ------------------------------------------------------------
# STEP 5. ĐỐI CHIẾU FRAUD SCORES
# ------------------------------------------------------------

pipeline_scores = reloaded_model.predict_proba(
    transformed_features
)[:, positive_class_reloaded]

np.testing.assert_allclose(
    pipeline_scores,
    reloaded_scores,
    rtol=0,
    atol=1e-8
)

# ------------------------------------------------------------
# STEP 6. ĐỐI CHIẾU ALERT DECISIONS
# ------------------------------------------------------------

pipeline_alerts = (
    pipeline_scores >= FINAL_THRESHOLD
).astype(np.int8)

assert np.array_equal(
    pipeline_alerts,
    reloaded_alerts
)

# ------------------------------------------------------------
# STEP 7. KIỂM TRA INPUT KHÔNG HỢP LỆ
# ------------------------------------------------------------

valid_record = records[0]

invalid_cases = {
    "Missing_Amount": {
        k: v for k, v in valid_record.items()
        if k != "amount"
    },
    "Negative_Amount": {
        **valid_record,
        "amount": -10
    },
    "Unknown_Transaction_Type": {
        **valid_record,
        "type": "UNKNOWN"
    }
}

for case_name, invalid_record in invalid_cases.items():

    try:
        fp.transform_for_inference(
            invalid_record,
            reloaded_preprocessor,
            expected_columns=list(X_verify.columns)
        )

    except ValueError:
        print(f"PASS: {case_name} rejected.")

    else:
        raise AssertionError(
            f"Invalid input accepted: {case_name}"
        )

# ------------------------------------------------------------
# STEP 8. KIỂM TRA ARTIFACT CHECKSUM
# ------------------------------------------------------------

with open(
    manifest_path,
    encoding="utf-8"
) as f:
    saved_manifest = json.load(f)

artifact_paths = {
    "model": final_model_path,
    "preprocessor": final_preprocessor_path,
    "feature_engineering": final_feature_module_path,
    "threshold": final_threshold_path
}

for name, path in artifact_paths.items():

    assert (
        sha256_file(path)
        == saved_manifest["sha256"][name]
    ), f"Checksum mismatch: {name}"

# ------------------------------------------------------------
# STEP 9. LƯU VERIFICATION REPORT
# ------------------------------------------------------------

verification_report = {
    "stage": "Model_Serialization_Verification",

    "selected_algorithm": "XGBoost",
    "selected_strategy": "Class_Weight",
    "selected_threshold": float(FINAL_THRESHOLD),

    "sample_dataset": "Validation_20pct",
    "verification_records": int(len(X_verify)),

    "raw_feature_count": 7,
    "engineered_feature_count": 9,
    "transformed_feature_count": 13,

    "maximum_model_reload_score_difference": (
        float(maximum_difference)
    ),

    "maximum_pipeline_score_difference": float(
        np.max(
            np.abs(
                pipeline_scores - reloaded_scores
            )
        )
    ),

    "feature_schema_verified": True,
    "serialized_model_reload_verified": True,
    "end_to_end_inference_verified": True,
    "alert_decisions_identical": True,
    "invalid_input_rejected": True,
    "artifact_checksums_verified": True,

    "model_retrained": False,
    "threshold_reoptimized": False,
    "test_used_for_selection": False,

    "verification_result": "PASS"
}

verification_path = (
    RESULT_DIR / "serialization_verification.json"
)

with open(
    verification_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        verification_report,
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# STEP 10. FINAL SUMMARY
# ------------------------------------------------------------

assert verification_path.is_file()

print("\n" + "=" * 60)
print("SER-03 — END-TO-END VERIFICATION")
print("=" * 60)

print("Raw features:", 7)
print("Engineered features:", 9)
print("Transformed features:", 13)

print("Verification records:", len(X_verify))

print(
    "Maximum pipeline score difference:",
    verification_report[
        "maximum_pipeline_score_difference"
    ]
)

print("Alert decisions identical: YES")
print("Input validation: PASS")
print("Artifact checksums: PASS")

print("\nSaved:", verification_path)

print("\nPASS: MODEL SERIALIZATION VERIFIED.")


PASS: Missing_Amount rejected.
PASS: Negative_Amount rejected.
PASS: Unknown_Transaction_Type rejected.

SER-03 — END-TO-END VERIFICATION
Raw features: 7
Engineered features: 9
Transformed features: 13
Verification records: 256
Maximum pipeline score difference: 0.0
Alert decisions identical: YES
Input validation: PASS
Artifact checksums: PASS

Saved: /content/drive/Shareddrives/M(anchine) L(earning)/Output/ML_Experiments/Final/results/serialization_verification.json

PASS: MODEL SERIALIZATION VERIFIED.
